# Main Focus of the Notebook: Exploratory Data Analysis and Dropping detected spots which are most likely noise

In [ ]:
from os import path
import pandas as pd
from IPython.display import display
from matplotlib import pyplot as plt
import napari
from skimage import io
import matplotlib.pyplot as plt
import seaborn as sns
import sys 
import zarr
import dask.array as da
import os 
sys.path.append('../src/')

from detections_preprocessing import hist_plot, box_whisker_plot

plt.rcParams["font.family"] = ""

### Do not change the code in the cell below 

In [ ]:
# This assumes that your notebook is inside 'Jupyter Notebooks', which is at the same level as 'test_data'
base_dir = os.path.join(os.path.dirname(os.path.abspath("__file__")), '..', 'test_data')

zarr_directory = 'zarr_file/all_channels_data'
zarr_full_path = os.path.join(base_dir, zarr_directory)

input_directory = 'datasets'
input_file_name = 'all_detections_channel3.pkl'
input_directory_full = os.path.join(base_dir,input_directory, input_file_name)

output_directory = 'datasets'
output_file_name = 'cleaned_spots_intensities_c3_all.pkl'
output_directory_full = os.path.join(base_dir,output_directory, output_file_name)

In [ ]:
spots_df = pd.read_pickle(input_directory_full)

z2 = zarr.open(zarr_full_path, mode='r')
z2.shape

In [ ]:
spots_df

In [ ]:
spots_df.shape

# Analysing Amplitude 
1. Box and Whisker Plot to find the upper and lower quartiles and caps 
2. Histogram to try and understand the distribution of the amplitude 

In [ ]:
stats = box_whisker_plot(dataframe = spots_df,column_name = 'amplitude')

In [ ]:
print(stats.keys())

In [ ]:
amplitude_upper_whisker = stats['upper_whisker']
print(f'The upper whisker for amplitude is at {amplitude_upper_whisker}')
amplitude_lower_whisker = stats['lower_whisker']
print(f'The upper whisker for amplitude is at {amplitude_lower_whisker}')

In [ ]:
print(len(spots_df[spots_df['amplitude'] > stats['upper_whisker']]))

**Use the below dataframe for visualisation in napari and determine the final cuttoff**

In [ ]:
high_amplitude_df = spots_df[spots_df['amplitude'] > stats['upper_whisker']].sort_values(by = 'amplitude', ascending = False)
high_amplitude_df.head()

In [ ]:
hist_plot(dataframe = spots_df, column_name = 'amplitude', bin_size = 10)

# Analysing Std Dev of X
1. Box and Whisker Plot to find the upper and lower quartiles and caps 
2. Histogram to try and understand the distribution of the sigma_x

In [ ]:
stats_sigma_x = box_whisker_plot(dataframe = spots_df,column_name = 'sigma_x')

In [ ]:
print(stats_sigma_x.keys())

In [ ]:
sigmax_upper_whisker = stats_sigma_x['upper_whisker']
sigmax_lower_whisker = stats_sigma_x['lower_whisker']

print(f'The upper whisker for sigma x is {sigmax_upper_whisker}')
print(f'The lower whisker for sigma x is {sigmax_lower_whisker}')

In [ ]:
print(stats_sigma_x.values())

In [ ]:
hist_plot(dataframe = spots_df, column_name = 'sigma_x', bin_size = 1, custom_xaxis = True, 
          lower_xlimit = 0, upper_xlimit = 20)

In [ ]:
high_sigmax_df = spots_df[spots_df['sigma_x'] > sigmax_upper_whisker].sort_values(by = 'sigma_x', ascending = False)
high_sigmax_df.head()

In [ ]:
high_sigmax_df.shape

In [ ]:
low_sigmax_df = spots_df[spots_df['sigma_x'] <= sigmax_lower_whisker].sort_values(by = 'sigma_x', ascending = False)
low_sigmax_df.head()

In [ ]:
low_sigmax_df.shape

# NOTES FOR SIGMA_X
# Two cases for sigma_x need to be tackled
1. **Spots with standard deviation of more than upper whisker**
2. **Spots with standard deviation of less than lower whisker**

# Analysing Std Dev of Y
1. Box and Whisker Plot to find the upper and lower quartiles and caps 
2. Histogram to try and understand the distribution of the sigma_y

In [ ]:
stats_sigma_y = box_whisker_plot(dataframe = spots_df,column_name = 'sigma_y')

In [ ]:
print(stats_sigma_y)

In [ ]:
hist_plot(dataframe = spots_df, column_name = 'sigma_y', bin_size = 1, custom_xaxis = True, 
          lower_xlimit = 0, upper_xlimit = 20)

In [ ]:
sigmay_upper_whisker = stats_sigma_y['upper_whisker']
sigmay_lower_whisker = stats_sigma_y['lower_whisker']

print(f'The upper whisker for sigma y is {sigmay_upper_whisker}')
print(f'The lower whisker for sigma y is {sigmay_lower_whisker}')

In [ ]:
high_sigmay_df = spots_df[spots_df['sigma_y']>sigmay_upper_whisker].sort_values(by='sigma_y', ascending = False)
high_sigmay_df.head()

In [ ]:
high_sigmay_df.shape

# Analysing Std Dev of Z
1. Box and Whisker Plot to find the upper and lower quartiles and caps 
2. Histogram to try and understand the distribution of the sigma_z

In [ ]:
stats_sigma_z = box_whisker_plot(dataframe = spots_df,column_name = 'sigma_z')

In [ ]:
print(stats_sigma_z)

In [ ]:
sigmaz_upper_whisker = stats_sigma_z['upper_whisker']
sigmaz_lower_whisker = stats_sigma_z['lower_whisker']

print(f'The upper whisker for sigma z is {sigmaz_upper_whisker}')
print(f'The lower whisker for sigma z is {sigmaz_lower_whisker}')

In [ ]:
print(len(spots_df[spots_df['sigma_z'] > sigmaz_upper_whisker]))

In [ ]:
high_sigmaz_df = spots_df[spots_df['sigma_z'] > sigmaz_upper_whisker]

In [ ]:
hist_plot(dataframe = spots_df, column_name = 'sigma_z', bin_size = 1,custom_xaxis = True, 
          lower_xlimit = 0, upper_xlimit = 20)

In [ ]:
spots_df['sigma_z'].describe()

In [ ]:
spots_df['sigma_y'].describe()

In [ ]:
spots_df['sigma_x'].describe()

# NAPARI VISUALISATION

# HIGH AMPLITUDE POINTS VISUALISATION

In [ ]:
# Create a napari viewer
viewer = napari.Viewer()

#access channel 3 only from zarr array 
dask_array = da.from_zarr(z2)

#the axis arrangement is (t,c,z,y,x)
#for the sake of improved performance only 1 channel could be imported here (if images get super large and performance issues occur)
all_channels = dask_array[:,:,:,:,:]

# Add the 4D stack to the viewer
layer_raw = viewer.add_image(all_channels, channel_axis = 1, name = ['actin', 'dynamin', 'clathrin'])
#other useful parameters 
#color_map = list
#contrast_limits = list of list 

# Add Bounding Box
layer_raw[0].bounding_box.visible = True
layer_raw[1].bounding_box.visible = True
layer_raw[2].bounding_box.visible = True

In [ ]:
points_layer = viewer.add_points(high_amplitude_df[["frame", "mu_z", "mu_y", "mu_x"]], size=3, 
                                name = 'High Amplitude Points', face_color = 'white', symbol = 'ring')

In [ ]:
high_amplitude_df['amplitude'].min()

## High Sigma X points analysis in Napari

In [ ]:
points_layer = viewer.add_points(high_sigmax_df[["frame", "mu_z", "mu_y", "mu_x"]], size=3, 
                                name = 'High Sigma X points', face_color = 'red', symbol = 'ring')

In [ ]:
#high_sigmax_df.iloc[972]

In [ ]:
points_layer = viewer.add_points(low_sigmax_df[["frame", "mu_z", "mu_y", "mu_x"]], size=3, 
                                name = 'Low Sigma X points', face_color = 'white', symbol = 'ring')

In [ ]:
#low_sigmax_df.iloc[1906]

# Testing for Sigma y in napari

In [ ]:
points_layer = viewer.add_points(high_sigmay_df[["frame", "mu_z", "mu_y", "mu_x"]], size=3, 
                                name = 'High Sigma Y points', face_color = 'white', symbol = 'ring')

# Testing for Sigma z in napari

In [ ]:
points_layer = viewer.add_points(high_sigmaz_df[["frame", "mu_z", "mu_y", "mu_x"]], size=3, 
                                name = 'High Sigma Z points', face_color = 'white', symbol = 'ring')

In [ ]:
#high_sigmaz_df.iloc[662]

# DROPPING VALUES OUTSIDE OF CUTOFF LIMITS

In [ ]:
print(f'the upper whisker for amplitude is {amplitude_upper_whisker}')
print(f'the lower whisker for amplitude is {amplitude_lower_whisker}')
print(f'the upper whisker for sigma x is {sigmax_upper_whisker}')
print(f'the lower whisker for sigma x is {sigmax_lower_whisker}')
print(f'the upper whisker for sigma y is {sigmay_upper_whisker}')
print(f'the lower whisker for sigma y is {sigmay_lower_whisker}')
print(f'the upper whisker for sigma z is {sigmaz_upper_whisker}')
print(f'the lower whisker for sigma z is {sigmaz_lower_whisker}')

## The above values are used to determine the cutoff values 
## Selected Cut off Values are 
1. Amplitude is 350 
2. Sigma x upper bound is 4
3. Sigma y upper bound is 4
4. Sigma z upper bound is 6

In [ ]:
# Define the conditions

# tolerance for limits 
tolerance = 1.0

#dropping spots above certain threshold 
condition_1 = spots_df['amplitude'] <= amplitude_upper_whisker + 31


#dropping spots above certain standard dev in x
condition_2 = spots_df['sigma_x'] <= sigmax_upper_whisker + 2


#dropping spots above certain standard dev in y 
condition_3 = spots_df['sigma_y'] <= sigmay_upper_whisker + 1


#dropping spots above certain standard dev in z
condition_4 = spots_df['sigma_z'] <= sigmaz_upper_whisker + 1 

#dropping spots below certain standard dev in x
condition_5 = spots_df['sigma_x'] >= sigmax_lower_whisker - 1

#dropping spots below certain standard dev in y
condition_6 = spots_df['sigma_y'] >= sigmay_lower_whisker 

#dropping spots below certain standard dev in z
condition_7 = spots_df['sigma_z'] >= sigmaz_lower_whisker 

#dropping spots out of bounds of z axis 
condition_8 = spots_df['mu_z'] >= 0
condition_9 = spots_df['mu_z'] <= z2.shape[2]

#dropping spots out of bounds of x axis 
condition_10 = spots_df['mu_x'] >= 0
condition_11 = spots_df['mu_x'] <= z2.shape[4]

#dropping spots out of bounds of y axis 
condition_12 = spots_df['mu_y'] >= 0
condition_13 = spots_df['mu_y'] <= z2.shape[3]


# Combine the conditions using logical AND (&)
cleaned_spots_df = spots_df[condition_1 & condition_2 & condition_3 & condition_4 & condition_5 & condition_6 &
condition_7 & condition_8 & condition_9 & condition_10 & condition_11 & condition_11 & condition_12 & condition_13].reset_index(drop = True)

# Display the resulting DataFrame
cleaned_spots_df

In [ ]:
dropped_spots = spots_df[~(condition_1 & condition_2 & condition_3 & condition_4 & condition_5 & condition_6 &
condition_7 & condition_8 & condition_9 & condition_10 & condition_11 & condition_11 & condition_12 & condition_13)]

In [ ]:
cleaned_spots_df['mu_z'].min()

In [ ]:
cleaned_spots_df.shape

In [ ]:
#Visualising the dropped spots and the cleaned spots 
points_layer = viewer.add_points(dropped_spots[["frame", "mu_z", "mu_y", "mu_x"]], size=3, 
                                name = 'Dropped Spots', face_color = 'blue', symbol = 'ring')

points_layer = viewer.add_points(cleaned_spots_df[["frame", "mu_z", "mu_y", "mu_x"]], size=3, 
                                name = 'Cleaned Spots', face_color = 'red', symbol = 'ring')

# Final Graphs

In [ ]:
# Set up subplots as a 2x2 grid
fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(10, 10))

# Flatten the axes array for easier indexing
axes = axes.flatten()

# Box plot for Amplitude
sns.boxplot(y='amplitude', data=cleaned_spots_df, ax=axes[0], showmeans=True, meanline=True, showfliers=False)
axes[0].set_title('Amplitude')

# Box plot for sigma_x
sns.boxplot(y='sigma_x', data=cleaned_spots_df, ax=axes[1], showmeans=True, meanline=True, showfliers=False)
axes[1].set_title('Sigma_x')

# Box plot for sigma_y
sns.boxplot(y='sigma_y', data=cleaned_spots_df, ax=axes[2], showmeans=True, meanline=True, showfliers=False)
axes[2].set_title('Sigma_y')

# Box plot for sigma_z
sns.boxplot(y='sigma_z', data=cleaned_spots_df, ax=axes[3], showmeans=True, meanline=True, showfliers=False)
axes[3].set_title('Sigma_z')

# Adjust layout
plt.tight_layout()

# Display the plot
plt.show()


In [ ]:
# Assuming cleaned_spots_df is your DataFrame with columns 'Amplitude', 'sigma_x', 'sigma_y', 'sigma_z'

# Set up subplots as a 2x2 grid
fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(10, 10))

# Flatten the axes array for easier indexing
axes = axes.flatten()

# Histogram for Amplitude with bins of size 50 starting from 180
axes[0].hist(cleaned_spots_df['amplitude'].dropna(), bins=range(180, int(cleaned_spots_df['amplitude'].max()) + 50, 50))
axes[0].set_title('Amplitude Histogram')

# Histogram for sigma_x with bins of size 1 starting from 0
axes[1].hist(cleaned_spots_df['sigma_x'].dropna(), bins=range(0, int(cleaned_spots_df['sigma_x'].max()) + 1, 1))
axes[1].set_title('Sigma_x Histogram')

# Histogram for sigma_y with bins of size 1 starting from 0
axes[2].hist(cleaned_spots_df['sigma_y'].dropna(), bins=range(0, int(cleaned_spots_df['sigma_y'].max()) + 1, 1))
axes[2].set_title('Sigma_y Histogram')

# Histogram for sigma_z with bins of size 1 starting from 0
axes[3].hist(cleaned_spots_df['sigma_z'].dropna(), bins=range(0, int(cleaned_spots_df['sigma_z'].max()) + 1, 1))
axes[3].set_title('Sigma_z Histogram')

# Adjust layout
plt.tight_layout()

# Display the plot
plt.show()


In [ ]:
# Set up subplots as a 2x2 grid
fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(10, 10))

# Flatten the axes array for easier indexing
axes = axes.flatten()

# Histogram for Amplitude with bins of size 50 starting from 180
ax = axes[0].hist(cleaned_spots_df['amplitude'], bins=range(175,375,10), edgecolor='black', density = True,alpha=0.7)
sns.kdeplot(cleaned_spots_df['amplitude'], ax=axes[0], color='red', linewidth=2, bw_method=1)
axes[0].set_title('Amplitude Histogram')

# Histogram for sigma_x with bins of size 1 starting from 0
axes[1].hist(cleaned_spots_df['sigma_x'], bins=range(0, int(cleaned_spots_df['sigma_x'].max()) + 1, 1), 
              density = True, edgecolor='black', alpha=0.7)
sns.kdeplot(cleaned_spots_df['sigma_x'], ax=axes[1], color='red', linewidth=2, bw_method=1)
axes[1].set_title('Sigma_x Histogram')

# Histogram for sigma_y with bins of size 1 starting from 0
axes[2].hist(cleaned_spots_df['sigma_y'], bins=range(0, int(cleaned_spots_df['sigma_y'].max()) + 1, 1), 
             density=True, edgecolor='black', alpha=0.7)
sns.kdeplot(cleaned_spots_df['sigma_y'], ax=axes[2], color='red', linewidth=2, bw_method=1)
axes[2].set_title('Sigma_y Histogram')

# Histogram for sigma_z with bins of size 1 starting from 0
axes[3].hist(cleaned_spots_df['sigma_z'], bins=range(0, int(cleaned_spots_df['sigma_z'].max()) + 1, 1), 
             density=True, edgecolor='black', alpha=0.7)
sns.kdeplot(cleaned_spots_df['sigma_z'], ax=axes[3], color='red', linewidth=2, bw_method=1)
axes[3].set_title('Sigma_z Histogram')

# Set x-axis limit for sigma_z KDE plot
axes[1].set_xlim(0, cleaned_spots_df['sigma_z'].max())
axes[2].set_xlim(0, cleaned_spots_df['sigma_z'].max())
axes[3].set_xlim(0, cleaned_spots_df['sigma_z'].max())

# Adjust layout
plt.tight_layout()

# Display the plot
plt.show()


In [ ]:
# Save the DataFrame to a Pickle file
cleaned_spots_df.to_pickle(output_directory_full)